# 🏠 Introduction to Databricks Lakehouse — Project Showcase

![Databricks](https://img.shields.io/badge/Platform-Databricks-FF3621?logo=databricks&logoColor=white)
![Unity Catalog](https://img.shields.io/badge/Governance-Unity%20Catalog-blueviolet)
![Delta Sharing](https://img.shields.io/badge/Sharing-Delta%20Sharing-00ADD8)
![Status](https://img.shields.io/badge/Status-Completed-brightgreen)

> **Platform:** DataCamp | **Track:** Associate Data Engineer in Databricks (Course 9)
> **Instructor:** Gang Wang

**Scenarios I worked through:** a junior data engineer at **Metro Transit Analytics** (taxi trip data in a medallion pipeline), **Meridian Analytics** (compute and notebooks), **NovaTech Solutions** (governance and Delta Sharing), **Summit Retail Corp** (Databricks Asset Bundles) and a final capstone at **Velocity Logistics**. All hands-on work was done in an Azure Databricks workspace.

**Objectives**
1. Explain why the lakehouse exists and how data is organized in it (medallion architecture, control vs data plane).
2. Choose and manage compute, and build multi-language notebooks.
3. Govern and share data with Unity Catalog, Delta Sharing and Lakehouse Federation.
4. Move from manual UI deployment to code with Databricks Asset Bundles.

Code is transcribed as `sql`, `python` and `yaml` blocks; screenshots in `./screenshots/` are kept only as evidence of results.

| Chapter | Topic |
|---|---|
| 1 | The Lakehouse Paradigm |
| 2 | Compute and Notebooks |
| 3 | Governance and Sharing |
| 4 | Deployment and Next Steps |

## Chapter 1 — The Lakehouse Paradigm

### 1.1 — Why the lakehouse?

| | Data lake | Data warehouse | Lakehouse |
|---|---|---|---|
| Data | any type, schema-on-read | structured only | any type |
| Cost | low | high | low |
| Reliability | no quality guarantees, risk of a "data swamp" | reliable, performant | reliable on top of open files |
| Governance | none built in | strong | unified (Unity Catalog) |
| Workloads | AI/data science, separate tools for BI | BI | analytics, AI and apps on one platform |

Databricks makes the lakehouse work with **open file formats (Delta Lake)**, **ACID transactions on the lake**, and **unified governance (Unity Catalog)**. Delta improves data quality with schema enforcement at write time, a transaction log that records every change, and time travel to query historical versions.

**Exercise — discovering lakehouse reliability features (Metro Transit Analytics).** I looked for evidence of what the lakehouse adds over the plain data lake the company used before:
- In the Catalog, `silver_taxi_trips` shows enforced types (`timestamp`, `double`, `int`) → **schema enforcement**.
- The table's transaction log lists every change with a version number and timestamp:

```sql
DESCRIBE HISTORY bronze_taxi_trips;
```

- The Details tab confirms the **Delta** format, which provides the ACID transactions and the log.

**Conclusion:** the lakehouse adds schema enforcement, a transaction log that tracks every change, and ACID transactions — all powered by Delta.

**Exercise — lakehouse vs warehouse trade-offs.** `bronze_taxi_trips` holds loosely typed raw data (every column a string, mixed formats, nulls) that a traditional warehouse would reject. The same platform also runs warehouse-style analytics on the clean silver layer:

```sql
SELECT pickup_zone,
       COUNT(*) AS trip_count
FROM silver_taxi_trips
GROUP BY pickup_zone
ORDER BY trip_count DESC
LIMIT 5;
```

![Top pickup zones by trip count from the silver layer](./screenshots/ch1_ex2_top_pickup_zones.png)

*Brooklyn leads with 670 trips, followed by Manhattan and Soho with 656 each.*

**Why not just move everything to the warehouse?** A warehouse can't store raw, semi-structured data efficiently, and scaling it for every workload would be expensive. The lakehouse handles raw ingestion and structured analytics on one platform, which also removes the duplication of maintaining two systems.

### 1.2 — The medallion architecture

Raw data from many sources arrives with missing fields, duplicates and mismatched types. The medallion architecture gives it a path from raw to ready in three layers:

| Layer | What it holds | Typical consumer |
|---|---|---|
| **Bronze** | data exactly as it arrived; append-only, nothing modified or deleted — the safety net to trace back to | data engineers |
| **Silver** | cleaned and validated: proper types, nulls and duplicates removed, schema enforced | engineers, analysts, data scientists |
| **Gold** | business-level aggregates optimized for dashboards and reports | analysts, business users |

These are common patterns, not strict rules: any role can read any layer if it needs to.

**Exercise — exploring the medallion layers.** Comparing the three tables' Overview tabs: bronze columns are all `string`, silver has `timestamp` and `double`, and gold has aggregated columns such as `avg_fare` and `trip_count`. Then I measured data quality directly:

```sql
SELECT 'bronze' AS layer,
       COUNT(*) AS null_count
FROM bronze_taxi_trips
WHERE passenger_count IS NULL
UNION ALL
SELECT 'silver',
       COUNT(*)
FROM silver_taxi_trips
WHERE passenger_count IS NULL;
```

![Null passenger counts in bronze vs silver](./screenshots/ch1_ex3_null_count_bronze_vs_silver.png)

*Bronze has 486 rows with a null `passenger_count`; silver has 0 — the cleaning step removed them.*

```sql
SELECT *
FROM gold_taxi_daily_summary
ORDER BY trip_count DESC
LIMIT 5;
```

![Pre-computed daily summaries in the gold layer](./screenshots/ch1_ex3_gold_daily_summary.png)

*Gold rows are already aggregated per date and zone (trip count, average and total fare), ready for a dashboard with no extra transformation.*

**Conclusion:** bronze stores raw data with its quality issues, silver cleans and validates it, and gold aggregates it into business-ready summaries.

### 1.3 — Platform architecture: control plane vs data plane

The course uses a hotel analogy:

| | Control plane — "the front desk" | Data plane — "your hotel room" |
|---|---|---|
| Managed by | Databricks | you, in your cloud account |
| Contains | workspace UI and notebooks, job scheduling and orchestration, cluster management | compute clusters, data in your cloud storage |
| Control | coordination of requests | you control access and networking |

The key point: **data never leaves your cloud environment**. Audit logs follow the same rule — they're stored as **system tables** in your own account and queried with SQL for compliance:

```sql
SELECT user_name, action_name, event_time
FROM system.access.audit
WHERE action_name = 'clusterCreate'
ORDER BY event_time DESC
LIMIT 5;
```

**Exercise — where are audit logs stored?** For a data-residency policy requiring logs to stay in the company's own infrastructure, I browsed the `system` catalog, found the audit table in the `access` schema, and inspected the captured events (timestamps, user IDs and action types):

```sql
SELECT *
FROM system.access.audit
LIMIT 10;
```

**Conclusion:** audit logs live as system tables in our own cloud account and are queried with standard SQL.

## Chapter 2 — Compute and Notebooks

### 2.1 — Cluster types and runtimes

A **cluster** is a set of virtual machines: one **driver** node coordinates the work and one or more **worker** nodes process it, reading data from cloud storage.

| | All-purpose cluster | Jobs cluster |
|---|---|---|
| Use | interactive development, ad-hoc queries | scheduled, automated production workloads |
| Lifecycle | runs until stopped manually | created when the job starts, terminated when it ends |
| Sharing | shared across a team | one task |
| Cost | pay even when idle ("a parked car") | cost-optimized |

A common, expensive mistake is running production pipelines on all-purpose clusters.

The **Databricks Runtime** is the versioned software stack on every cluster: Apache Spark, Delta Lake, Python, R and Scala. **LTS** (long-term support) versions are recommended for production, and the **ML Runtime** adds pre-installed ML libraries.

**Exercise — creating and configuring a cluster (Meridian Analytics).** I walked through the creation form for a `patient_analysis` personal compute cluster on the latest LTS runtime, reviewing the node type (VM size and memory) and the auto-termination timeout, then cancelled without creating it.

**Conclusion:** the runtime version determines the pre-installed versions of Spark, Delta Lake and the bundled libraries.

### 2.2 — Managing and configuring clusters

- **Finding clusters:** All Clusters, My Clusters, Shared with Me, filterable by state (Running, Terminated).
- **Termination is not deletion:** configuration and notebooks survive; in-memory data, caches and pip-installed packages are cleared.
- **Restart** when a new library isn't loading, the cluster behaves unpredictably, cached data must be cleared, or an init script / environment variable changed.
- **Autoscaling** adds or removes workers with demand; **auto-termination** shuts down idle clusters so forgotten ones don't burn budget.
- **Cluster policies** are admin guardrails: users create clusters only within the allowed bounds.

```json
{
  "max_workers": 8,
  "auto_termination_minutes": 30,
  "allowed_node_types": [
    "Standard_DS3_v2",
    "Standard_DS4_v2"
  ]
}
```

**Exercise — managing a shared cluster.** I reviewed the `meridian_shared_etl` cluster used for daily ETL: its *Terminate after* setting, node type and runtime version, focusing on how auto-termination controls cost.

**Conclusion:** after termination the configuration is preserved, but in-memory data is lost.

### 2.3 — Notebook fundamentals

A Databricks notebook is an interactive document of runnable cells, attached to a cluster, that mixes code, results and documentation. It has a default language, and **magic commands** switch the language of a single cell:

| Magic | Purpose |
|---|---|
| `%python` / `%sql` / `%scala` / `%r` | run code in that language |
| `%md` | render Markdown |
| `%sh` | run shell commands |
| `%run <path>` | run another notebook in the same context, so its functions and variables become available |

```python
# Load shared helper functions, then use them
%run /Shared/utils/data_helpers
clean_df = clean_nulls(raw_df)
```

Results render inline: SQL queries as interactive tables, DataFrames with built-in visualization options, and errors with stack traces and line numbers.

**Exercise — building a multi-language notebook.** In a new Python notebook attached to compute, I combined the three cell types a data-quality pipeline needs:

```python
# Cell 1 — default language (Python)
df = spark.table("silver_taxi_trips")
display(df)
```

![Python cell loading the silver table as a DataFrame](./screenshots/ch2_ex3_python_cell_spark_table.png)

*`spark.table()` returns a PySpark DataFrame with a typed schema (`trip_id: string`, `pickup_datetime: timestamp`, …) rendered as a table.*

```sql
%sql
SELECT COUNT(*) AS total_trips FROM silver_taxi_trips
```

![SQL magic cell counting trips](./screenshots/ch2_ex3_sql_magic_count.png)

*The silver table has **9,519 trips**. The result is also exposed to Python as the `_sqldf` DataFrame.*

```markdown
%md
## Trip Data Summary
```

![Markdown magic cell rendered as a heading](./screenshots/ch2_ex3_md_magic_render.png)

*The `%md` cell renders as formatted documentation inside the same notebook.*

**Conclusion:** a magic command changes only its own cell — the next new cell goes back to the notebook's default language (Python).

### 2.4 — Collaboration and version control

Notebooks are shared with three permission levels:

| Permission | View | Execute | Modify |
|---|---|---|---|
| Can Read | ✅ | ❌ | ❌ |
| Can Run | ✅ | ✅ | ❌ |
| Can Edit | ✅ | ✅ | ✅ |

Cell comments support code review, and the built-in **version history** shows who changed what and when — but it's linear, with no branching or merging. **Databricks Repos** (Git folders) brings full Git integration into the workspace (GitHub, GitLab, Bitbucket, Azure DevOps): clone, commit, push, pull, branch and merge, enabling CI/CD with pull requests and automated deployment.

**Exercise — exploring notebook version history.** I added a cell with `# TODO: add data quality checks` to the previous notebook, saved it, and checked the version list in the history panel, where each save appears with its timestamp.

**Conclusion:** the limitation of built-in history compared with Repos is that it's linear — no branching support.

## Chapter 3 — Governance and Sharing

### 3.1 — Unity Catalog and lineage

With hundreds of tables across teams, two questions come up constantly: *who can see this data?* and *where does this number come from?* Unity Catalog centralizes both answers.

**Hierarchy:** **metastore** (one per account) → **catalog** (e.g. `production`, `development`) → **schema** (e.g. `sales`) → tables, views and functions. Objects are addressed with the three-level name `catalog.schema.table`.

**Access control** uses SQL grants at any level, inherited downward (a grant on a catalog applies to all its schemas):

```sql
-- Grant read access to a schema
GRANT SELECT
ON SCHEMA production.sales
TO `analytics_team`;

-- Revoke table-level access
REVOKE SELECT
ON TABLE production.sales.customers
FROM `intern_group`;
```

**Lineage** is tracked automatically as queries run: upstream (where data comes from), downstream (what depends on a table), impact analysis (what breaks if I change a column) and debugging (trace a wrong report back to its source).

**Exercise — Unity Catalog hierarchy (NovaTech Solutions).** I expanded catalogs → schemas → tables in the Catalog explorer and verified the naming convention:

```sql
SELECT * FROM default.client_engagement_gold LIMIT 5;
```

![Querying a gold table with its schema-qualified name](./screenshots/ch3_ex1_three_level_namespace_query.png)

*The gold table returns engagement aggregated by region (average engagement and client count).*

**Conclusion:** the **schema** level directly contains tables, views and functions.

**Exercise — exploring table metadata.** Before an analyst used `client_engagement_silver` in a report, I checked its columns and types on the Overview tab and its format and storage location on the Details tab.

![Details tab of the silver table showing its storage location](./screenshots/ch3_ex2_table_details_storage.png)

*Unity Catalog shows creation time, creator and the `abfss://` storage location in Azure Data Lake Storage.*

**Conclusion:** beyond schemas, Unity Catalog tracks lineage, access permissions and table properties such as storage format and location.

### 3.2 — Delta Sharing fundamentals

Traditional sharing (export, email, upload) creates stale copies with no control once the file leaves. **Delta Sharing** gives recipients **live, read-only access with no copies**, and access can be revoked instantly. The course analogy: instead of photocopying a book, you issue a library card that can be cancelled at any time.

A **share** is a named collection of tables (usually clean gold-layer data), and a **recipient** is the organization or person receiving it, who gets an activation link to set up access:

```sql
CREATE SHARE partner_data;

ALTER SHARE partner_data
ADD TABLE production.sales.gold_daily_revenue;

CREATE RECIPIENT acme_corp;

GRANT SELECT ON SHARE partner_data
TO RECIPIENT acme_corp;
```

**Exercise — creating a Delta Share.** From `client_engagement_gold`, I created the share `partner_engagement_share` through the UI and registered the partner `apex_consulting` as a recipient — without copying any data.

![Share with the Apex Consulting recipient](./screenshots/ch3_ex3_share_recipient.png)

*The share lists `apex_consulting` as its recipient with token authentication.*

**Conclusion:** when the partner queries the shared table, the data stays in our cloud storage and is read live.

**Exercise — adding a table to an existing share.** The partner also needed the silver data, so I added `client_engagement_silver` to the same share instead of creating a new one — one share per partner, with tables added as needs grow.

![Adding a table to an existing share](./screenshots/ch3_ex4_add_to_existing_share.png)

*The share dialog with "Add Table to an existing share" selected.*

![Share now containing gold and silver tables](./screenshots/ch3_ex4_share_with_two_tables.png)

*Both `client_engagement_gold` and `client_engagement_silver` are listed in `partner_engagement_share`.*

**Conclusion:** the recipient gets access to the newly added table automatically, with no action on their side.

### 3.3 — Delta Sharing: types and trade-offs

| | Native sharing | Open protocol sharing |
|---|---|---|
| Recipient | another Databricks customer | any platform: Spark, pandas, Power BI, Snowflake, Tableau |
| Setup | minimal — create the share and grant access | recipient installs a Delta Sharing client |
| Governance | Unity Catalog on both sides | Unity Catalog on the provider side |

With open protocol, the recipient reads the data with a credentials profile, for example from pandas:

```python
import delta_sharing

profile = "config.share"
client = delta_sharing.SharingClient(profile)
tables = client.list_all_tables()

df = delta_sharing.load_as_pandas(f"{profile}#share.schema.table")
```

**Costs:** the same region is typically free of egress charges; cross-region transfers are charged by the cloud provider; cross-cloud (e.g. Azure → AWS) has the highest egress fees. Shared data should sit close to its main recipients.

**Exercise — native vs open protocol sharing.** A new partner, AnalyticaBI, uses Snowflake, so I created an **Open** recipient on the existing share with a one-day token expiry and located its activation link.

![Creating an Open (non-Databricks) recipient](./screenshots/ch3_ex5_open_recipient.png)

*The recipient type choice: Databricks (native) vs Open (external platform).*

![Activation link generated for the open recipient](./screenshots/ch3_ex5_activation_link.png)

*Token-based authentication and the activation link the partner uses to connect.*

**Conclusion:** the partner uses the activation link to download a credentials file that configures their Delta Sharing client to read data from our storage.

**Exercise — cross-cloud sharing costs.** Our workspace is on Azure US East; the partner GlobalTech is on AWS EU West. Reviewing the share, the recipient's cloud and region and the table's row count made the main cost driver clear: **cloud egress charges** for data read across providers and regions.

### 3.4 — Lakehouse Federation

When data is spread across PostgreSQL, MySQL, Snowflake or SQL Server, copying all of it is slow and creates duplicates. **Lakehouse Federation** queries external databases in place:

```sql
-- 1. Create a connection
CREATE CONNECTION partner_pg
TYPE postgresql
OPTIONS (
  host 'db.partner.com',
  port '5432',
  user 'reader',
  password secret('scope', 'key')
);

-- 2. Create a foreign catalog
CREATE FOREIGN CATALOG partner_db
USING CONNECTION partner_pg;

-- 3. Query the external table
SELECT *
FROM partner_db.public.orders
LIMIT 10;
```

Federated tables appear in the Unity Catalog hierarchy with the same access control and lineage as local tables, and can be joined with lakehouse tables in a single query. Credentials come from a secret scope rather than plain text.

| | Federate | Ingest |
|---|---|---|
| Access | real-time | batch copy |
| Query volume | low | high |
| Latency | depends on the source | low |
| Compliance | no copies | data moves |
| Transforms | limited | full medallion pipeline |

**Rule of thumb:** federate for real-time, low-volume or compliance-restricted access; ingest for high-volume, repeated, performance-critical workloads.

**Exercise — setting up a federated connection.** I ordered the setup steps for querying an external MySQL database:
1. Gather hostname, port and credentials.
2. Create a federation connection.
3. Create a foreign catalog that maps to the external database.
4. Browse the external tables through the Unity Catalog browser.
5. Join local and federated tables in a query.

## Chapter 4 — Deployment and Next Steps

### 4.1 — From manual to automated deployment

Configuring jobs one at a time in the UI and copying notebooks to production folders leads to settings drifting between staging and production, no way to rebuild a deployment from scratch, configuration knowledge that leaves with a team member, and no answer when an audit asks what changed last month.

**Databricks Asset Bundles (DABs)** put everything — code, jobs, pipelines and environments — in one project folder, versioned in Git and deployed with a single CLI command.

| | Traditional | DAB |
|---|---|---|
| Configuration | UI clicks | YAML files in Git |
| Reproducibility | manual, error-prone | automated, consistent |
| Environment promotion | copy and reconfigure | `bundle deploy --target prod` |
| Rollback | recreate from memory | `git revert` + redeploy |
| Collaboration | one person at a time | full Git workflow |

### 4.2 — Inside an asset bundle

```text
my_project/
|-- databricks.yml        # central config: bundle, targets, resources
|-- src/                  # notebooks and code
|   |-- etl_pipeline.py
|   |-- data_quality.py
|-- resources/            # job and pipeline definitions
|   |-- jobs/
|   |   |-- nightly_etl.yml
|   |-- pipelines/
|       |-- sales_pipeline.yml
|-- tests/                # optional tests
    |-- test_etl.py
```

**Targets** map to environments (dev, staging, production); **resources** define jobs, pipelines and dashboards. The CLI covers the lifecycle:

```bash
databricks bundle validate                        # check config for errors
databricks bundle deploy --target production      # deploy to a target
databricks bundle run nightly_etl                 # trigger a deployed job
databricks bundle destroy                         # remove deployed resources
```

The deployment workflow: develop and validate locally → commit to Git → CI/CD deploys to staging and runs tests → promote to production.

### 4.3 — Exercises with the `retail-etl-bundle` (Summit Retail Corp)

**Examining the project structure.** The cloned bundle contains a `resources/` folder, a `src/` folder with the `bronze_to_silver` and `silver_to_gold` notebooks, and the central `databricks.yml`:

```yaml
bundle:
  name: retail-etl-bundle

targets:
  dev:
    workspace:
      host: ${workspace_url}
      root_path: /Shared/retail-etl/dev
    default: true
  production:
    workspace:
      host: ${workspace_url}
      root_path: /Shared/retail-etl/production

resources:
  jobs:
    nightly_etl:
      name: "Nightly Retail ETL"
      schedule:
        quartz_cron_expression: "0 0 2 * * ?"
        timezone_id: "America/New_York"
      tasks:
        - task_key: bronze_to_silver
          notebook_task:
            notebook_path: ./src/bronze_to_silver.py
          new_cluster:
            spark_version: "17.1.x-scala2.13"
            node_type_id: "Standard_F4s"
            num_workers: 1
        - task_key: silver_to_gold
          depends_on:
            - task_key: bronze_to_silver
          notebook_task:
            notebook_path: ./src/silver_to_gold.py
          new_cluster:
            spark_version: "17.1.x-scala2.13"
            node_type_id: "Standard_F4s"
            num_workers: 1
```

What this file tells me:
- **`dev` is the default target**; `production` uses the same workspace with a separate root path.
- One job, `nightly_etl`, runs every day at **02:00 New York time** (Quartz cron `0 0 2 * * ?`).
- Its two tasks follow the medallion flow, and `depends_on` makes `silver_to_gold` wait for `bronze_to_silver` to finish.
- Each task runs on a new **jobs cluster** (1 worker), which exists only for the run.

**DAB vs traditional deployment.** The Jobs page in the workspace was empty, while `databricks.yml` already defined a complete job ready to deploy with one command — code and configuration live together instead of being managed separately. **Key advantage:** the configuration is version-controlled, reproducible and gives a full audit trail.

**Inspecting and modifying the bundle.** I added a `staging` target so deployments can be tested before production:

```yaml
  staging:
    workspace:
      host: https://adb-staging.azuredatabricks.net
      root_path: /Shared/retail-etl/staging
```

In the lab I first typed the host as `${https://adb-staging.azuredatabricks.net}`. The `${...}` syntax is for variable references (like `${workspace_url}` in the other targets), so a literal URL must go without it — exactly the kind of mistake validation should catch.

Before deploying, `databricks bundle validate` checks `databricks.yml` for syntax errors, missing fields and configuration problems, so issues are caught before anything reaches an environment.

**Tracing a job from config to code.** Job task definitions — including the notebook each task runs — live in the **`resources.jobs`** section of `databricks.yml`. Each `notebook_path` points to a file in `src/`, and both notebooks exist in the project.

### 4.4 — Capstone: end-to-end lakehouse workflow (Velocity Logistics)

The final scenario brings the four chapters together for a nightly shipping-data pipeline:

1. **Compute:** a scheduled nightly ETL needs no interactive work, so the right choice is a **jobs cluster** on an **LTS runtime**, with autoscaling (e.g. 2–6 workers) for variable volume.
2. **Data:** compare the raw `default.shipping_bronze` with the cleansed, aggregated `default.shipping_gold`. Row counts should drop as quality increases (raw → cleaned → aggregated).
3. **Governance:** in the Catalog, review `shipping_gold` columns, types, storage format, location and owner, and trace its lineage upstream.
4. **Deployment:** review `databricks.yml` in `logistics-pipeline-bundle` for the targets and the nightly job.

**Scheduling and deploying it:** use a jobs cluster and deploy with `databricks bundle deploy --target production` — a jobs cluster terminates when the job finishes, and the bundle makes the deployment reproducible, versioned and auditable.

**Balancing security, cost and governance.** To share `shipping_gold` with a partner on a different cloud: confirm the data's storage location and owner, then use **open protocol Delta Sharing** so the partner reads the data live from our storage under Unity Catalog governance — no copies leave our account, and keeping the shared data in the gold layer limits the volume (and egress cost) read across clouds.

## 🧠 Key Takeaways

- **The lakehouse unifies the lake and the warehouse:** open Delta files with ACID transactions, schema enforcement, a transaction log and time travel, governed by Unity Catalog.
- **Medallion layers make quality measurable:** bronze keeps the raw truth, silver cleans it, gold serves it — and comparing layers (null counts, row counts) is a quick quality check.
- **Your data stays in your cloud:** the control plane is managed by Databricks; compute, storage and audit logs live in the customer's account.
- **Match compute to the workload:** all-purpose for interactive work, jobs clusters for production; LTS runtimes, autoscaling, auto-termination and policies control cost.
- **Notebooks mix languages** with magic commands, but serious version control needs Repos/Git folders.
- **Govern once, share live:** Unity Catalog grants and lineage, Delta Sharing (native or open) without copies, and Federation to query external databases in place.
- **Deploy as code:** Asset Bundles make jobs reproducible across dev → staging → production, with `bundle validate` before every deploy.

## 💼 Application in a Data Analyst / Data Engineer Role

- **Pipeline design:** structuring ingestion as bronze → silver → gold, and validating each hop with simple SQL checks.
- **Cost-aware operations:** choosing jobs clusters for scheduled ETL and setting auto-termination so idle compute doesn't drain budget.
- **Data governance:** granting least-privilege access by schema, using lineage for impact analysis before changing a column, and answering compliance questions from `system.access.audit`.
- **External data products:** delivering gold tables to partners through Delta Sharing instead of CSV exports, with cross-cloud egress in mind.
- **Production readiness:** versioning job definitions in a bundle, promoting through environments, and rolling back with `git revert` + redeploy.

## ✅ Skills Demonstrated

- Comparing data lakes, warehouses and lakehouses, and explaining Delta Lake's reliability features (`DESCRIBE HISTORY`, schema enforcement, ACID).
- Validating medallion layers with SQL (`UNION ALL` null-count comparisons, gold aggregates).
- Explaining the control plane / data plane split and querying audit logs in `system.access.audit`.
- Configuring all-purpose vs jobs clusters, runtimes, autoscaling, auto-termination and cluster policies.
- Building multi-language Databricks notebooks with `spark.table()`, `display()`, `%sql`, `%md` and `%run`.
- Navigating Unity Catalog's three-level namespace, reading table metadata, and writing `GRANT` / `REVOKE` statements.
- Creating Delta Shares and recipients (native and open protocol), adding tables to shares, and reasoning about egress costs.
- Setting up Lakehouse Federation (connection → foreign catalog → query) and deciding between federation and ingestion.
- Reading and editing a Databricks Asset Bundle (`databricks.yml` targets, jobs, task dependencies, Quartz schedules) and using the bundle CLI.